# Notebook 4 — Evaluation, Confusion Matrix & Feature Importance

**Module:** 7006SCN Machine Learning and Big Data — Coventry University  
**Project:** Scalable Fake News Detection Using Distributed ML on Common Crawl  

---

## Objectives
1. Evaluate all 3 models on the **held-out TEST set** (unbiased estimate).
2. Compute: Accuracy, F1, Precision, Recall, ROC-AUC.
3. Plot confusion matrices.
4. Extract & visualise feature importance (Random Forest).
5. Generate ROC curve data.
6. Export all results for Tableau dashboards.

In [ ]:
# ── 4.1  Bootstrap ──────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from config.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark import StorageLevel
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator, MulticlassClassificationEvaluator
)

spark = get_spark()
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

TABLEAU_DIR = Path("../tableau")
TABLEAU_DIR.mkdir(exist_ok=True)

In [ ]:
# ── 4.2  Load test set & saved models ───────────────────────────────────
from pyspark.ml.classification import (
    LogisticRegressionModel, LinearSVCModel, RandomForestClassificationModel
)

test_df = spark.read.parquet("../data/parquet/features/test").persist(StorageLevel.MEMORY_AND_DISK)
print(f"Test set: {test_df.count():,} rows")

MODELS_DIR = Path("../data/models")

models = {
    "LogisticRegression": LogisticRegressionModel.load(str(MODELS_DIR / "LogisticRegression_mllib")),
    "LinearSVC":          LinearSVCModel.load(str(MODELS_DIR / "LinearSVC_mllib")),
    "RandomForest":       RandomForestClassificationModel.load(str(MODELS_DIR / "RandomForest_mllib")),
}

print(f"Loaded {len(models)} models")

In [ ]:
# ── 4.3  Full evaluation on TEST set ────────────────────────────────────

auc_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
acc_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="accuracy"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)
precision_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedPrecision"
)
recall_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="weightedRecall"
)

test_results = []
predictions_dict = {}   # store for confusion matrix

for name, model in models.items():
    preds = model.transform(test_df)
    predictions_dict[name] = preds
    
    accuracy  = acc_eval.evaluate(preds)
    f1        = f1_eval.evaluate(preds)
    precision = precision_eval.evaluate(preds)
    recall    = recall_eval.evaluate(preds)
    
    try:
        auc = auc_eval.evaluate(preds)
    except Exception:
        auc = None
    
    test_results.append({
        "model":     name,
        "accuracy":  round(accuracy, 4),
        "f1":        round(f1, 4),
        "precision": round(precision, 4),
        "recall":    round(recall, 4),
        "auc":       round(auc, 4) if auc else None,
    })

results_df = pd.DataFrame(test_results)
print(results_df.to_string(index=False))

In [ ]:
# ── 4.4  Confusion Matrices ─────────────────────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, preds) in zip(axes, predictions_dict.items()):
    # Collect predictions to driver (small test set — safe)
    pdf = preds.select("label", "prediction").toPandas()
    cm = confusion_matrix(pdf["label"], pdf["prediction"], labels=[0, 1])
    
    disp = ConfusionMatrixDisplay(cm, display_labels=["Reliable", "Fake"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"{name}\nAcc={results_df.loc[results_df['model']==name, 'accuracy'].values[0]}")

plt.tight_layout()
plt.savefig(str(TABLEAU_DIR / "confusion_matrices.png"), dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved confusion_matrices.png")

In [ ]:
# ── 4.5  ROC Curve Data ─────────────────────────────────────────────────
from sklearn.metrics import roc_curve, auc

roc_data_all = []

fig, ax = plt.subplots(figsize=(8, 6))

for name, preds in predictions_dict.items():
    # Get probability of class 1 (if available)
    if "probability" in preds.columns:
        # Extract P(class=1) from the probability vector
        pdf = (
            preds
            .select("label", F.element_at("probability", 2).alias("prob_1"))
            .toPandas()
        )
    else:
        # LinearSVC: use rawPrediction as score
        from pyspark.ml.functions import vector_to_array
        pdf = (
            preds
            .select(
                "label",
                vector_to_array("rawPrediction").getItem(1).alias("prob_1")
            )
            .toPandas()
        )
    
    fpr, tpr, thresholds = roc_curve(pdf["label"], pdf["prob_1"])
    roc_auc = auc(fpr, tpr)
    
    ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})")
    
    # Collect for Tableau export
    for f, t in zip(fpr, tpr):
        roc_data_all.append({"model": name, "fpr": f, "tpr": t, "auc": roc_auc})

ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Test Set")
ax.legend()
plt.tight_layout()
plt.savefig(str(TABLEAU_DIR / "roc_curves.png"), dpi=150, bbox_inches="tight")
plt.show()

# Export ROC data for Tableau
roc_df = pd.DataFrame(roc_data_all)
roc_df.to_csv(str(TABLEAU_DIR / "roc_data.csv"), index=False)
print(f"✓ Exported {len(roc_df):,} ROC data points for Tableau")

In [ ]:
# ── 4.6  Feature Importance (Random Forest) ─────────────────────────────
rf_model = models["RandomForest"]

# Feature importance vector (length = numFeatures from HashingTF)
importances = rf_model.featureImportances.toArray()

# Top 30 most important feature indices
top_k = 30
top_indices = np.argsort(importances)[::-1][:top_k]
top_scores  = importances[top_indices]

fi_df = pd.DataFrame({
    "feature_index": top_indices,
    "importance": top_scores,
    "rank": range(1, top_k + 1)
})

# Since we used HashingTF, indices don't map directly to words.
# For interpretability, we add the hash bucket label.
fi_df["feature_label"] = fi_df["feature_index"].apply(lambda x: f"hash_{x}")

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=fi_df, x="importance", y="feature_label", ax=ax, palette="viridis")
ax.set_title("Top 30 Features by Importance (Random Forest)")
ax.set_xlabel("Gini Importance")
plt.tight_layout()
plt.savefig(str(TABLEAU_DIR / "feature_importance.png"), dpi=150, bbox_inches="tight")
plt.show()

# Export for Tableau
fi_df.to_csv(str(TABLEAU_DIR / "feature_importance.csv"), index=False)
print("✓ Exported feature_importance.csv")

In [ ]:
# ── 4.7  Reverse hash lookup (Optional — map hash indices to words) ─────
# If you want to know WHICH words correspond to top hash indices,
# re-apply Tokenizer + StopWordsRemover on a sample and build a
# reverse index from the HashingTF.

from pyspark.ml import PipelineModel
from pyspark.ml.feature import HashingTF
from collections import defaultdict

# Load the saved feature pipeline
feat_pipeline = PipelineModel.load("../data/models/feature_pipeline")

# Get a sample of raw text data
sample_df = spark.read.parquet("../data/parquet/common_crawl_articles").limit(5000)

# Apply tokenizer + stopwords remover (first 2 stages)
tokenized = feat_pipeline.stages[0].transform(sample_df)
filtered  = feat_pipeline.stages[1].transform(tokenized)

# Build reverse hash map: hash_index → set of words
NUM_FEATURES = 2**18
from pyspark.ml.feature import HashingTF as HF
from pyspark.sql.functions import explode

words_df = filtered.select(explode("filtered_words").alias("word"))
unique_words = [row.word for row in words_df.distinct().collect()]

hash_to_word = defaultdict(list)
for w in unique_words:
    h = hash(w) % NUM_FEATURES
    if h in set(top_indices):
        hash_to_word[h].append(w)

# Enrich feature importance with actual words
fi_df["words"] = fi_df["feature_index"].apply(
    lambda x: ", ".join(hash_to_word.get(x, ["unknown"])[:5])
)
fi_df.to_csv(str(TABLEAU_DIR / "feature_importance_with_words.csv"), index=False)
print(fi_df[["rank", "importance", "words"]].head(15).to_string(index=False))

In [ ]:
# ── 4.8  Export final test metrics for Tableau ──────────────────────────
results_df.to_csv(str(TABLEAU_DIR / "test_metrics.csv"), index=False)
print("✓ Exported test_metrics.csv")

# Summary of all Tableau exports from this pipeline:
print("\n=== Tableau export inventory ===")
for f in sorted(TABLEAU_DIR.glob("*")):
    size = f.stat().st_size / 1024
    print(f"  {f.name:45s} {size:>8.1f} KB")

In [ ]:
# ── Cleanup ──
test_df.unpersist()
# spark.stop()